In [23]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *
from lib.kpi_processor.KPIReportEurope import *
from datetime import date
from datetime import timedelta

In [25]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = KPIReport.kpi_list + KPIEmissionSource.kpi_list

In [26]:
cursor.execute("DROP VIEW IF EXISTS Country_Yearly_KPI;")
cursor.execute("DROP VIEW IF EXISTS Customer_Yearly_KPI;")
conn.commit()


In [27]:
query = """
CREATE VIEW IF NOT EXISTS Country_Yearly_KPI AS
SELECT
    kd.Year,
    kc.Name AS Country,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
INNER JOIN KPI_Country kc ON kd.CustomerId = kc.CountryId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year, kd.CustomerId, kc.Name;"""
cursor.execute(query)

query = """
CREATE VIEW IF NOT EXISTS Customer_Yearly_KPI AS
SELECT
    kd.Year,
    kc.Name AS CustomerName,
    kc.Country AS Country,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
INNER JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year,kd.CustomerId, kc.Name;"""
cursor.execute(query)

conn.commit()

In [28]:
query = "SELECT * FROM Country_Yearly_KPI WHERE Year = 2026;"
query_customer = "SELECT * FROM Customer_Yearly_KPI WHERE Year = 2026;"
country_df = pd.read_sql_query(query, conn)
customer_df = pd.read_sql_query(query_customer, conn)
conn.close()
display(country_df)
display(customer_df)


,Year,Country,FOVMain,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,ServicePipeKm,ServicePipeCoveredKm,ReportCount,...,Bm2Density,NGDensity,PGDensity,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2026,United Kingdom,94.27,83944.28,78815.21,81884.29,77190.46,2060.00,1624.76,2628.0,...,0.13,1.09,0.19,19.91,1.02,68.88,10.19,75.51,12.86,11.63
1,2026,Italy,93.13,210761.54,196275.09,209849.03,195428.51,912.51,846.57,4382.0,...,0.22,0.67,0.51,13.39,0.49,67.75,18.37,42.73,32.26,25.00
2,2026,Romania,94.66,183.72,174.11,162.63,153.94,21.10,20.17,7.0,...,0.49,1.11,1.68,17.08,1.65,63.58,17.70,38.88,58.52,2.61
3,2026,Switzerland,97.57,178.63,174.14,125.69,122.63,52.94,51.51,3.0,...,0.03,0.57,0.17,32.81,1.56,61.72,3.91,73.88,21.64,4.48
4,2026,Greece,99.35,17216.23,17111.27,14784.50,14688.91,2431.73,2422.37,414.0,...,0.27,0.77,0.50,10.89,0.44,67.82,20.85,45.99,30.20,23.81
5,2026,Germany,92.55,35652.49,33071.35,24428.24,22607.65,11224.34,10463.79,1558.0,...,0.04,0.10,0.15,15.98,0.93,67.69,15.40,17.01,26.31,56.68
6,2026,Poland,38.41,4379.15,1681.98,4379.15,1681.98,0.00,0.00,120.0,...,0.04,0.25,0.25,39.78,2.03,50.78,7.41,31.45,31.53,37.02
7,2026,Austria,39.64,185.12,65.20,146.00,57.87,39.13,7.33,42.0,...,0.03,0.05,0.21,0.00,0.00,88.24,11.76,7.89,36.84,55.26
8,2026,Ireland,95.27,4420.82,4190.69,4225.02,4025.13,195.81,165.56,131.0,...,0.02,0.13,0.05,26.69,1.36,63.14,8.81,38.10,15.46,46.44
9,2026,Netherlands,94.14,1073.16,1009.69,710.45,668.79,362.71,340.90,38.0,...,0.06,0.17,0.36,27.72,2.62,58.80,10.86,13.98,28.92,57.11


,Year,CustomerName,Country,FOVMain,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,ServicePipeKm,ServicePipeCoveredKm,...,Bm2Density,NGDensity,PGDensity,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2026,Wales and West Utilities,United Kingdom,92.97,2667.13,2479.60,2667.13,2479.60,0.00,0.00,...,0.18,1.53,0.40,22.88,1.38,66.49,9.26,73.65,19.24,7.11
1,2026,APRETIGAS,Italy,91.01,15441.73,14060.35,15156.47,13794.36,285.26,265.98,...,0.21,0.75,0.48,16.09,0.57,66.70,16.64,43.55,27.94,28.51
2,2026,ADRIGAS,Italy,95.54,774.91,739.87,765.20,731.09,9.71,8.78,...,0.08,0.44,0.43,29.68,2.78,58.73,8.81,50.85,49.15,0.00
3,2026,M Reti,Italy,99.94,84.40,84.35,84.40,84.35,0.00,0.00,...,0.57,1.74,1.77,12.50,0.00,71.28,16.22,32.81,33.26,33.93
4,2026,CPL CONCORDIA,Romania,94.66,183.72,174.11,162.63,153.94,21.10,20.17,...,0.49,1.11,1.68,17.08,1.65,63.58,17.70,38.88,58.52,2.61
5,2026,RETEGAS BARI spa,Italy,96.17,170.15,163.63,170.15,163.63,0.00,0.00,...,0.35,1.04,0.68,17.38,0.35,62.06,20.21,52.15,34.36,13.50
6,2026,ASTEA,Italy,95.36,177.86,169.61,177.86,169.61,0.00,0.00,...,0.14,0.95,0.48,23.05,1.65,65.43,9.88,62.40,31.78,5.81
7,2026,AIL,Switzerland,97.57,178.63,174.14,125.69,122.63,52.94,51.51,...,0.03,0.57,0.17,32.81,1.56,61.72,3.91,73.88,21.64,4.48
8,2026,Centria,Italy,98.68,212.17,209.37,212.17,209.37,0.00,0.00,...,0.14,0.62,0.35,10.40,0.00,75.25,14.36,43.43,24.58,31.99
9,2026,DEPA,Greece,99.35,17216.23,17111.27,14784.50,14688.91,2431.73,2422.37,...,0.27,0.77,0.50,10.89,0.44,67.82,20.85,45.99,30.20,23.81


In [29]:
country_df[['Country', 'LisaDensity']].max()

Country        United Kingdom
LisaDensity              2.79
dtype: object